In [ ]:
import fitz #install PyMuPDF
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.schema.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import faiss

In [ ]:
#Clip Model
import os

from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPEN_API_KEY")

## initialize the clip model for inified embeddings

clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32") #used for formatting
clip_model.eval()



In [ ]:
#Emnedding function

def embed_image(image_data):
    if isinstance(image_data,str):
        image=Image.open(image_data).convert("RGB")
    else:
        image=image_data

    inputs=clip_processor(images=image,return_tensors="pt")
    with torch.no_grad():
        features=clip_model.get_image_features(**inputs)
        features=features/features.norm(dim=1,keepdim=True)
        return features.squeeze().numpy()
    
#same clip model for text embedding as well
def embed_text(text):


    inputs=clip_processor(text=text,return_tensors="pt",padding=True,truncation=True,max_lenth=77)
    with torch.no_grad():
        features=clip_model.get_text_features(**inputs)
        features=features/features.norm(dim=1,keepdim=True)
        return features.squeeze().numpy()

In [ ]:
#process pdf
pdf_path="multimodel_sample.pdf"
doc=fitz.open(pdf_path)
all_docs=[]
all_embeddings=[]
image_data_store={}

#text_splitter

splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
